In [1]:
import pandas as pd

pancreatic = pd.read_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/GSE14245_series_matrix.csv")
breast = pd.read_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/GSE20266_series_matrix.csv")
lung = pd.read_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/GSE32175_series_matrix.csv")
gastric = pd.read_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/GSE64951_series_matrix.csv")

In [2]:
def cancer_healthy_split(df):
    cancer_samples = df["ID_REF"]
    healthy_samples = df["ID_REF"]
    for column in df.columns:
        if "cancer" in column.lower():
            cancer_samples =pd.concat([cancer_samples, df[column]], axis=1)
        else:
            healthy_samples = pd.concat([healthy_samples, df[column]], axis=1)
    return cancer_samples, healthy_samples

pancreatic_cancer, pancreatic_healthy = cancer_healthy_split(pancreatic)
breast_cancer, breast_healthy = cancer_healthy_split(breast)    
lung_cancer, lung_healthy = cancer_healthy_split(lung)
gastric_cancer, gastric_healthy = cancer_healthy_split(gastric)

In [3]:
print(type(lung_cancer))

<class 'pandas.core.frame.DataFrame'>


In [4]:
def mean_express_and_diff(cancer_df, healthy_df):
    mean_express_cancer = cancer_df.mean(axis=1, numeric_only=True)
    mean_express_healthy = healthy_df.mean(axis=1, numeric_only=True)
    diff = mean_express_cancer - mean_express_healthy
    return mean_express_cancer, mean_express_healthy, diff

panc_mean_exp_can, panc_mean_exp_he, panc_diff = mean_express_and_diff(pancreatic_cancer, pancreatic_healthy)
breast_mean_exp_can, breast_mean_exp_he, breast_diff = mean_express_and_diff(breast_cancer, breast_healthy)
lung_mean_exp_can, lung_mean_exp_he, lung_diff = mean_express_and_diff(lung_cancer, lung_healthy)
gastric_mean_exp_can, gastric_mean_exp_he, gastric_diff = mean_express_and_diff(gastric_cancer, gastric_healthy)

In [5]:
print(gastric_diff)

0        -22.171570
1         13.605586
2        225.758586
3        -39.408462
4          3.724441
            ...    
54670     -9.542376
54671     86.189909
54672    129.422841
54673     93.459543
54674     34.903357
Length: 54675, dtype: float64


In [6]:
from scipy.stats import ttest_ind

panc_t_stat, panc_p_value = ttest_ind(pancreatic_cancer.drop(["ID_REF"], axis=1), pancreatic_healthy.drop(["ID_REF"], axis=1), axis=1, equal_var=False)
breast_t_stat, breast_p_value = ttest_ind(breast_cancer.drop(["ID_REF"], axis=1), breast_healthy.drop(["ID_REF"], axis=1), axis=1, equal_var=False)
lung_t_stat, lung_p_value = ttest_ind(lung_cancer.drop(["ID_REF"], axis=1), lung_healthy.drop(["ID_REF"], axis=1), axis=1, equal_var=False)
gastric_t_stat, gastric_p_value = ttest_ind(gastric_cancer.drop(["ID_REF"], axis=1), gastric_healthy.drop(["ID_REF"], axis=1), axis=1, equal_var=False)

In [7]:
def combine_df(df, mean_exp_cancer, mean_exp_healthy, diff, p_value):
    df_results = pd.concat([df["ID_REF"], mean_exp_cancer, mean_exp_healthy, diff, p_value], axis=1)
    df_results.columns = ["ID_REF", "mean_expression_cancer", "mean_expression_healthy", "difference", "p_value"]
    df_results["regulated"] = df_results.apply(lambda row: "upregulated" if row["difference"] > 0 else ("downregulated" if row["difference"] < 0 else "no_change"), axis=1)
    df_results = df_results.sort_values(by="p_value", ascending=True)
    return df_results

panc_results = combine_df(pancreatic, panc_mean_exp_can, panc_mean_exp_he, panc_diff, pd.Series(panc_p_value))
breast_results = combine_df(breast, breast_mean_exp_can, breast_mean_exp_he, breast_diff, pd.Series(breast_p_value))
lung_results = combine_df(lung, lung_mean_exp_can, lung_mean_exp_he, lung_diff, pd.Series(lung_p_value))
gastric_results = combine_df(gastric, gastric_mean_exp_can, gastric_mean_exp_he, gastric_diff, pd.Series(gastric_p_value))

In [8]:
annotation = pd.read_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/gene_symbols/GPL570-55999.csv")
print(annotation.head(5))

          ID  GB_ACC SPOT_ID Species Scientific Name Annotation Date  \
0  1007_s_at  U48705     NaN            Homo sapiens      10/06/2014   
1    1053_at  M87338     NaN            Homo sapiens      10/06/2014   
2     117_at  X51757     NaN            Homo sapiens      10/06/2014   
3     121_at  X69699     NaN            Homo sapiens      10/06/2014   
4  1255_g_at  L36861     NaN            Homo sapiens      10/06/2014   

       Sequence Type                  Sequence Source  \
0  Exemplar sequence  Affymetrix Proprietary Database   
1  Exemplar sequence                          GenBank   
2  Exemplar sequence  Affymetrix Proprietary Database   
3  Exemplar sequence                          GenBank   
4  Exemplar sequence  Affymetrix Proprietary Database   

                                  Target Description Representative Public ID  \
0  U48705 /FEATURE=mRNA /DEFINITION=HSU48705 Huma...                   U48705   
1  M87338 /FEATURE= /DEFINITION=HUMA1SBU Human re...          

C:\Users\USER\AppData\Local\Temp\ipykernel_10920\2702940227.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  annotation = pd.read_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/gene_symbols/GPL570-55999.csv")


In [9]:
annotation = annotation[["ID", "Gene Symbol"]]
results = [panc_results, breast_results, lung_results, gastric_results]

for idx in range(len(results)):
    results[idx] = results[idx].merge(annotation, left_on="ID_REF", right_on="ID")
    results[idx] = results[idx].drop(["ID"], axis=1)
    results[idx].insert(1, "Gene Symbol", results[idx].pop("Gene Symbol"))
    results[idx] = results[idx].dropna(subset=["Gene Symbol"])
    results[idx] = results[idx].sort_values("p_value")
    results[idx] = results[idx].drop_duplicates(subset=["Gene Symbol"], keep="first")


panc_results, breast_results, lung_results, gastric_results = results
print(panc_results.tail(5))

            ID_REF    Gene Symbol  mean_expression_cancer  \
54651  221670_s_at           LHX3              198.384167   
54659    241572_at          PDZD9                1.101667   
54664  201695_s_at            PNP                1.910833   
54665  222304_x_at        OR7E47P                1.028333   
54666   1561431_at  RP11-284F21.8                1.685000   

       mean_expression_healthy    difference   p_value    regulated  
54651               198.361667  2.250000e-02  0.999518  upregulated  
54659                 1.101667  2.220446e-16  1.000000  upregulated  
54664                 1.910833  0.000000e+00  1.000000    no_change  
54665                 1.028333  0.000000e+00  1.000000    no_change  
54666                 1.685000  0.000000e+00  1.000000    no_change  


In [10]:
panc_top_15_up = panc_results[(panc_results["p_value"] <= 0.05) & (panc_results["regulated"] == "upregulated")].head(15)
breast_top_15_up = breast_results[(breast_results["p_value"] <= 0.05) & (breast_results["regulated"] == "upregulated")].head(15)
lung_top_15_up = lung_results[(lung_results["p_value"] <= 0.05) & (lung_results["regulated"] == "upregulated")].head(15)
gastric_top_15_up = gastric_results[(gastric_results["p_value"] <= 0.05) & (gastric_results["regulated"] == "upregulated")].head(15)

panc_top_15_down = panc_results[(panc_results["p_value"] <= 0.05) & (panc_results["regulated"] == "downregulated")].head(15)
breast_top_15_down = breast_results[(breast_results["p_value"] <= 0.05) & (breast_results["regulated"] == "downregulated")].head(15)
lung_top_15_down = lung_results[(lung_results["p_value"] <= 0.05) & (lung_results["regulated"] == "downregulated")].head(15)
gastric_top_15_down = gastric_results[(gastric_results["p_value"] <= 0.05) & (gastric_results["regulated"] == "downregulated")].head(15)

panc_lung_up = set(panc_top_15_up["Gene Symbol"]) & set(lung_top_15_up["Gene Symbol"])
panc_breast_up = set(panc_top_15_up["Gene Symbol"]) & set(breast_top_15_up["Gene Symbol"])
panc_gastric_up = set(panc_top_15_up["Gene Symbol"]) & set(gastric_top_15_up["Gene Symbol"])
breast_lung_up = set(breast_top_15_up["Gene Symbol"]) & set(lung_top_15_up["Gene Symbol"])
breast_gastric_up = set(breast_top_15_up["Gene Symbol"]) & set(gastric_top_15_up["Gene Symbol"])
lung_gastric_up = set(lung_top_15_up["Gene Symbol"]) & set(gastric_top_15_up["Gene Symbol"])
common_all_up = set(panc_top_15_up["Gene Symbol"]) & set(breast_top_15_up["Gene Symbol"]) & set(lung_top_15_up["Gene Symbol"]) & set(gastric_top_15_up["Gene Symbol"])

panc_lung_down = set(panc_top_15_down["Gene Symbol"]) & set(lung_top_15_down["Gene Symbol"])
panc_breast_down = set(panc_top_15_down["Gene Symbol"]) & set(breast_top_15_down["Gene Symbol"])
panc_gastric_down = set(panc_top_15_down["Gene Symbol"]) & set(gastric_top_15_down["Gene Symbol"])
breast_lung_down = set(breast_top_15_down["Gene Symbol"]) & set(lung_top_15_down["Gene Symbol"])
breast_gastric_down = set(breast_top_15_down["Gene Symbol"]) & set(gastric_top_15_down["Gene Symbol"])
lung_gastric_down = set(lung_top_15_down["Gene Symbol"]) & set(gastric_top_15_down["Gene Symbol"])
common_all_down = set(panc_top_15_down["Gene Symbol"]) & set(breast_top_15_down["Gene Symbol"]) & set(lung_top_15_down["Gene Symbol"]) & set(gastric_top_15_down["Gene Symbol"])

print("Common top 100 genes between pancreatic and lung cancer (upregulated):", panc_lung_up)
print("Common top 100 genes between pancreatic and breast cancer (upregulated):", panc_breast_up)
print("Common top 100 genes between pancreatic and gastric cancer (upregulated):", panc_gastric_up)
print("Common top 100 genes between breast and lung cancer (upregulated):", breast_lung_up)
print("Common top 100 genes between breast and gastric cancer (upregulated):", breast_gastric_up)
print("Common top 100 genes between lung and gastric cancer (upregulated):", lung_gastric_up)
print("Common top 100 genes across all cancers (upregulated):", common_all_up)

print("\nCommon top 100 genes between pancreatic and lung cancer (downregulated):", panc_lung_down)
print("Common top 100 genes between pancreatic and breast cancer (downregulated):", panc_breast_down)
print("Common top 100 genes between pancreatic and gastric cancer (downregulated):", panc_gastric_down)
print("Common top 100 genes between breast and lung cancer (downregulated):", breast_lung_down)
print("Common top 100 genes between breast and gastric cancer (downregulated):", breast_gastric_down)
print("Common top 100 genes between lung and gastric cancer (downregulated):", lung_gastric_down)
print("Common top 100 genes across all cancers (downregulated):", common_all_down)

Common top 100 genes between pancreatic and lung cancer (upregulated): set()
Common top 100 genes between pancreatic and breast cancer (upregulated): set()
Common top 100 genes between pancreatic and gastric cancer (upregulated): {'PAPPA'}
Common top 100 genes between breast and lung cancer (upregulated): set()
Common top 100 genes between breast and gastric cancer (upregulated): set()
Common top 100 genes between lung and gastric cancer (upregulated): set()
Common top 100 genes across all cancers (upregulated): set()

Common top 100 genes between pancreatic and lung cancer (downregulated): set()
Common top 100 genes between pancreatic and breast cancer (downregulated): set()
Common top 100 genes between pancreatic and gastric cancer (downregulated): set()
Common top 100 genes between breast and lung cancer (downregulated): set()
Common top 100 genes between breast and gastric cancer (downregulated): set()
Common top 100 genes between lung and gastric cancer (downregulated): set()
Comm

In [11]:
def combine_names(up, down):
    combined = pd.concat([up, down], axis=0)
    return combined.tolist()

panc_combined = combine_names(panc_top_15_up["Gene Symbol"], panc_top_15_down["Gene Symbol"])
lung_combined = combine_names(lung_top_15_up["Gene Symbol"], lung_top_15_down["Gene Symbol"])
breast_combined = combine_names(breast_top_15_up["Gene Symbol"], breast_top_15_down["Gene Symbol"])
gastric_combined = combine_names(gastric_top_15_up["Gene Symbol"], gastric_top_15_down["Gene Symbol"])
print(panc_combined)

['EME2', 'CREB3L2', 'HOOK1', 'HINT1', 'DTX3', 'H2AFX', 'SEC14L5', 'SOCS5', 'PAPPA', 'ZNF503', 'ARIH1', 'USH2A', 'ATAD3B', 'SEC11C', 'KRAS', 'CDKL3', 'CD7', 'PNPLA8', 'TPBGL', 'FAHD1', 'HIST1H3E', 'SLC4A8', 'CKAP2 /// IGLC1 /// IGLJ2 /// IGLJ2 /// IGLJ3 /// IGLJ3 /// IGLJ3 /// IGLV3-1 /// IGLV3-1 /// IGLV@', 'SEC23IP', 'WDR48', 'TRIM14', 'C4orf27', 'PAGE4', 'SMUG1', 'DHRS9']


In [15]:
def orig_gene_and_filter(orig_data, annotation, top_combined):
    orig_data = orig_data.merge(annotation, left_on="ID_REF", right_on="ID")
    orig_data = orig_data.drop(["ID"], axis=1)
    orig_data.insert(1, "Gene Symbol", orig_data.pop("Gene Symbol"))
    orig_data = orig_data.dropna(subset=["Gene Symbol"])
    orig_data = orig_data.drop_duplicates(subset=["Gene Symbol"], keep="first")
    
    filtered_df = orig_data[orig_data["Gene Symbol"].isin(top_combined)]
    filtered_df = filtered_df.T
    filtered_df.columns = filtered_df.loc["Gene Symbol"]
    filtered_df = filtered_df.drop(["ID_REF", "Gene Symbol"])
    keep_cols = [col for col in filtered_df.columns if len(col) <= 15]
    filtered_df = filtered_df[keep_cols]
    filtered_df.columns.name = None
    return filtered_df

panc_filtered = orig_gene_and_filter(pancreatic, annotation, panc_combined)
lung_filtered = orig_gene_and_filter(lung, annotation, lung_combined)
breast_filtered = orig_gene_and_filter(breast, annotation, breast_combined)
gastric_filtered = orig_gene_and_filter(gastric, annotation, gastric_combined)

In [16]:
panc_filtered

,SLC4A8,HINT1,EME2,C4orf27,ARIH1,PAPPA,SEC23IP,KRAS,CDKL3,TRIM14,...,DHRS9,HOOK1,WDR48,DTX3,SEC11C,PNPLA8,ATAD3B,FAHD1,ZNF503,TPBGL
pancreatic cancer saliva-1,32.59,2.69,9.7,0.01,2.54,344.09,2.1,0.0,0.86,24.89,...,43.73,27.52,0.0,0.0,2.0,1.57,2.69,0.0,0.28,82.55
pancreatic cancer saliva-2,21.38,4.71,6.35,0.0,0.0,426.81,0.0,0.0,6.0,0.12,...,103.07,2.77,1.06,0.0,0.51,0.0,0.01,3.75,1.91,17.99
pancreatic cancer saliva-3,10.66,1.46,4.37,0.01,0.0,273.02,8.56,0.0,3.75,10.6,...,26.79,26.48,5.62,0.0,3.23,4.68,1.77,1.63,0.73,33.15
pancreatic cancer saliva-4,21.14,2.47,8.89,0.26,0.69,415.06,0.68,0.0,0.01,36.0,...,37.87,9.96,0.0,2.01,2.51,0.0,1.2,1.94,0.9,66.75
pancreatic cancer saliva-5,13.0,4.75,7.44,0.0,2.25,465.64,0.0,3.01,0.0,9.43,...,162.32,0.0,0.19,0.0,6.13,0.0,1.22,3.1,0.04,59.11
pancreatic cancer saliva-6,2.53,2.24,4.84,1.14,2.35,268.81,3.7,2.65,0.0,3.77,...,140.28,0.01,11.64,0.0,1.79,0.0,3.96,0.0,0.0,92.79
pancreatic cancer saliva-7,3.78,5.75,0.08,0.04,12.5,274.08,3.21,9.15,2.07,44.5,...,270.4,0.04,0.0,0.0,6.12,0.0,3.96,0.45,0.0,0.25
pancreatic cancer saliva-8,11.83,2.97,6.33,0.0,0.15,243.8,0.0,0.01,0.0,13.13,...,69.73,15.89,0.25,0.0,3.48,0.0,2.98,0.02,0.0,99.21
pancreatic cancer saliva-9,0.5,2.57,0.01,0.0,3.32,179.98,6.85,2.89,0.01,1.59,...,181.37,0.04,5.08,0.54,0.66,12.43,0.99,0.0,0.0,35.72
pancreatic cancer saliva-10,29.35,5.31,0.01,0.01,1.57,226.92,0.0,0.57,3.68,0.0,...,115.06,0.01,5.34,1.51,3.76,2.06,2.45,1.52,0.24,73.01


In [17]:
print(f"Shape of filtered pancreatic data: {panc_filtered.shape}")
print(f"First 5 Pancreatic data samples:\n{panc_filtered.head(5)} ")

Shape of filtered pancreatic data: (24, 29)
First 5 Pancreatic data samples:
                           SLC4A8 HINT1  EME2 C4orf27 ARIH1   PAPPA SEC23IP  \
pancreatic cancer saliva-1  32.59  2.69   9.7    0.01  2.54  344.09     2.1   
pancreatic cancer saliva-2  21.38  4.71  6.35     0.0   0.0  426.81     0.0   
pancreatic cancer saliva-3  10.66  1.46  4.37    0.01   0.0  273.02    8.56   
pancreatic cancer saliva-4  21.14  2.47  8.89    0.26  0.69  415.06    0.68   
pancreatic cancer saliva-5   13.0  4.75  7.44     0.0  2.25  465.64     0.0   

                            KRAS CDKL3 TRIM14  ...   DHRS9  HOOK1 WDR48  DTX3  \
pancreatic cancer saliva-1   0.0  0.86  24.89  ...   43.73  27.52   0.0   0.0   
pancreatic cancer saliva-2   0.0   6.0   0.12  ...  103.07   2.77  1.06   0.0   
pancreatic cancer saliva-3   0.0  3.75   10.6  ...   26.79  26.48  5.62   0.0   
pancreatic cancer saliva-4   0.0  0.01   36.0  ...   37.87   9.96   0.0  2.01   
pancreatic cancer saliva-5  3.01   0.0   9.

In [18]:
panc_filtered.to_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/processed/panc_filtered.csv", index=False)
lung_filtered.to_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/processed/lung_filtered.csv", index=False)
breast_filtered.to_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/processed/breast_filtered.csv", index=False)
gastric_filtered.to_csv("C:/My Work/Python/Machine Learning/Projects/Salivary Biomarker Research/data/processed/gastric_filtered.csv", index=False)